# AMEX Enterprise Credit Risk Platform
## Notebook 31 — Phase 2, Problem 3: Expected Credit Loss (IFRS9/CECL) — ECL Modeling
### Problem Statement 3 of 14 (notebook 2 of 4)

CRISP-DM stage: **Modeling**. Depends on Notebook 30 (`ecl_policy.json`), Problem 1's real champion model + preprocessing artifacts (Notebook 05), and Problem 4's real severity-tier assignments (Notebook 27's `severity_scores_holdout.csv`).

**What happens in this notebook, in order:**

1. Score real PD for every holdout customer, live, using Problem 1's actual saved champion model (same load-and-score pattern Notebook 08 already used) — never assumed or reloaded from a cache.
2. Join Problem 4's real severity tier for the same holdout customers (validated as a clean 1-to-1 match — this is the same holdout split Problem 1's Notebook 04 produced, reused unmodified by every downstream notebook).
3. Assign IFRS9 stages using **only** the real PD and real tier — never the known outcome label. The label is used **once, afterward**, purely to validate that stages rank-order with real default rates (Section 10) — the exact shortcut Notebook 08 took and flagged in itself, not repeated here.
4. Apply the macro-overlay scenario weights and discount to present value, both per Notebook 30's policy.
5. Compute **both** IFRS9 (staged: 12-month ECL for Stage 1, lifetime ECL for Stage 2/3) and **CECL** (lifetime ECL for the entire portfolio, no staging) from the identical inputs, then compare both against Notebook 08's flat-LGD baseline.

**Deliverables:** `ecl_by_customer.csv`, `ecl_stage_summary.csv`, `ecl_standard_comparison.csv`, 2 charts, `notebook_31_summary.json`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 30's POLICY + PROBLEM 1/4's REAL RESULTS
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 30's Policy + Problem 1/4's Real Results")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P1_ROOT = PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
P1_ARTIFACTS = P1_ROOT / "artifacts"
# Legacy pre-rename data-cache folder -- the large engineered/raw CSVs were deliberately excluded
# from the Phase 1 folder reorg copy and confirmed still living under this old folder name.
P1_LEGACY_ROOT = P1_ROOT.parent / "Problem 1 Credit Default_Probability of Default"
P4_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "04_Problem4_Delinquency_Escalation_Loss_Severity"
P3_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "03_Problem3_Expected_Credit_Loss_IFRS9_CECL"
ARTIFACTS_DIR = P3_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PILLAR_DIRS = {
    "p3_policy": P3_ROOT / "01_ECL_Policy",
    "p3_modeling": P3_ROOT / "02_ECL_Modeling",
    "p3_validation_deployment": P3_ROOT / "03_Validation_Deployment",
    "p3_reporting_packaging": P3_ROOT / "04_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = P1_ARTIFACTS / "project_config.json"
NB04_SUMMARY_PATH = P1_ARTIFACTS / "notebook_04_summary.json"
NB05_SUMMARY_PATH = P1_ARTIFACTS / "notebook_05_summary.json"
NB08_SUMMARY_PATH = P1_ARTIFACTS / "notebook_08_summary.json"
ECL_POLICY_PATH = PILLAR_DIRS["p3_policy"] / "ecl_policy.json"
P4_SCORES_PATH = P4_ROOT / "02_LGD_Modeling" / "severity_scores_holdout.csv"
for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (NB04_SUMMARY_PATH, "run Problem 1's Notebook 04 first."),
    (NB05_SUMMARY_PATH, "run Problem 1's Notebook 05 first."),
    (NB08_SUMMARY_PATH, "run Problem 1's Notebook 08 first."),
    (ECL_POLICY_PATH, "run Notebook 30 (this problem's Notebook 1) first."),
    (P4_SCORES_PATH, "run Problem 4's Notebook 27 first -- this notebook needs its real severity tiers."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(ECL_POLICY_PATH, "r", encoding="utf-8") as f:
    ECL_POLICY = json.load(f)

RANDOM_SEED = P1_CONFIG["random_seed"]
_resource_limits = P1_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or P1_CONFIG.get("warp_thread_count")
    or P1_CONFIG["hardware"]["logical_cores_detected"]
)

P1_PILLAR_DIRS = {k: Path(v) for k, v in P1_CONFIG["pillar_dirs"].items()}
MODEL_DEV_DIR = P1_PILLAR_DIRS["model_development"]
MODELS_SUBDIR = MODEL_DEV_DIR / "models"
PREPROCESSING_PATH = MODELS_SUBDIR / "preprocessing_artifacts.joblib"
CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_MODEL_PATH = MODELS_SUBDIR / f"{CHAMPION_NAME}.joblib"


def _resolve_engineered_file(filename: str) -> Path:
    """Same 3-candidate resolver used by Problem 4's Notebooks 27/28 (deterministic new-structure
    path, legacy pre-rename folder, summary-JSON path as last resort) -- a candidate must be real
    data (>10KB) to count, ruling out a stray placeholder/note file of the same name."""
    _candidates = [
        P1_ROOT / "Feature_Engineering" / filename,
        P1_LEGACY_ROOT / "Feature_Engineering" / filename,
        Path(NB04_SUMMARY["output_files"][filename]),
    ]
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > 10_000:
            return _c
    raise FileNotFoundError(
        f"{filename} not found (as real data, >10KB) at any checked location:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: re-run Problem 1's Notebook 04 from its current Phase1_Foundation location."
    )


TEST_SPLIT_ENG_PATH = _resolve_engineered_file("test_split_engineered.csv")

for _p, _fix in [(CHAMPION_MODEL_PATH, "re-run Problem 1's Notebook 05."),
                  (PREPROCESSING_PATH, "re-run Problem 1's Notebook 05.")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

print(f"Champion PD model (Problem 1, real)      : {CHAMPION_NAME}")
print(f"LGD by tier (Problem 4, real)             : {ECL_POLICY['lgd_by_tier']['values']}")
print(f"EAD/account (ASSUMPTION, inherited)       : ${ECL_POLICY['ead_per_account_usd']:,}")
print(f"Staging rubric (Notebook 30, outcome-free): SICR={ECL_POLICY['staging_criteria']['sicr_pd_multiple']}x "
      f"avg PD or Moderate/Severe tier; Stage 3=Severe tier AND PD>"
      f"{ECL_POLICY['staging_criteria']['stage3_pd_threshold']}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
np.random.seed(RANDOM_SEED)
print(f"WARP_THREAD_COUNT: {WARP_THREAD_COUNT}")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD CHAMPION MODEL & PREPROCESSING ARTIFACTS
# =============================================================================
_section("SECTION 3: Load Champion Model & Preprocessing Artifacts")

_t0 = time.time()
champion_model = joblib.load(CHAMPION_MODEL_PATH)
preprocessing_artifacts = joblib.load(PREPROCESSING_PATH)
print(f"Loaded champion model '{CHAMPION_NAME}' ({time.time() - _t0:.1f}s)")

label_encoders = preprocessing_artifacts["label_encoders"]
feature_medians = preprocessing_artifacts["feature_medians"]
scaler = preprocessing_artifacts["scaler"]
all_feature_cols = preprocessing_artifacts["all_feature_cols"]
categorical_encode_cols = preprocessing_artifacts["categorical_encode_cols"]
numeric_feature_cols = preprocessing_artifacts["numeric_feature_cols"]
champion_uses_scaled = CHAMPION_NAME == "logistic_regression"

print(f"Feature columns loaded: {len(all_feature_cols)} "
      f"({len(numeric_feature_cols)} numeric + {len(categorical_encode_cols)} categorical)")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD HOLDOUT DATA, APPLY SAVED PREPROCESSING, SCORE REAL PD (MEASURED)
# =============================================================================
_section("SECTION 4: Load Holdout Data, Apply Saved Preprocessing, Score Real PD")

# --- Same explicit-schema Polars pattern as Notebook 08. Every PD used below is
#     scored HERE, live, by the real saved champion model on the real holdout
#     split -- never assumed or reloaded from a cache. ---
SPLIT_CSV_SCHEMA = {"customer_ID": pl.Utf8, "target": pl.Int8}
for _c in categorical_encode_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Utf8
for _c in numeric_feature_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Float32

_t0 = time.time()
holdout_pl = pl.read_csv(str(TEST_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
print(f"Loaded test_split_engineered.csv: {holdout_pl.shape[0]:,} x {holdout_pl.shape[1]} ({time.time() - _t0:.1f}s)")

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in numeric_feature_cols
]
holdout_pl = holdout_pl.with_columns(_inf_clean_exprs)
for c in categorical_encode_cols:
    holdout_pl = holdout_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    _mapping = {cat: i for i, cat in enumerate(label_encoders[c]["classes"])}
    holdout_pl = holdout_pl.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in numeric_feature_cols]
holdout_pl = holdout_pl.with_columns(_impute_exprs)

X_holdout = holdout_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_holdout = holdout_pl.get_column("target").to_numpy().astype(np.int64, copy=False)
holdout_customer_ids = holdout_pl.get_column("customer_ID").to_numpy()

_train_mean = scaler["mean"]
_train_std = scaler["std"]
X_holdout_scaled = (X_holdout - _train_mean) / _train_std
Xc_holdout = X_holdout_scaled if champion_uses_scaled else X_holdout

_t0 = time.time()
PD_12M = champion_model.predict_proba(Xc_holdout)[:, 1].astype(np.float64)
print(f"Scored real PD for {len(PD_12M):,} holdout customers in {time.time() - _t0:.1f}s "
      f"(MEASURED -- champion model '{CHAMPION_NAME}', this run)")
print(f"PD distribution: min {PD_12M.min():.4f}, mean {PD_12M.mean():.4f}, max {PD_12M.max():.4f}")
print(f"Real observed holdout default rate: {y_holdout.mean():.4%}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: JOIN PROBLEM 4's REAL SEVERITY TIER (VALIDATED 1-TO-1 MATCH)
# =============================================================================
_section("SECTION 5: Join Problem 4's Real Severity Tier (Validated 1-to-1 Match)")

pd_df = pd.DataFrame({"customer_ID": holdout_customer_ids, "pd_12m": PD_12M, "target": y_holdout})
p4_scores = pd.read_csv(P4_SCORES_PATH)[["customer_ID", "severity_tier", "target"]].rename(
    columns={"target": "p4_target"})

ecl_df = pd_df.merge(p4_scores, on="customer_ID", how="inner", validate="one_to_one")
if len(ecl_df) != len(pd_df):
    raise RuntimeError(
        f"Join dropped customers: {len(pd_df):,} scored here vs {len(ecl_df):,} matched to Problem 4's "
        "real severity tiers. Problem 3 and Problem 4 must be scoring the identical holdout split.\n"
        "Fix: re-run Problem 1's Notebook 04 once, then re-run Problem 4's Notebook 27 and this notebook."
    )
if not (ecl_df["target"] == ecl_df["p4_target"]).all():
    raise RuntimeError(
        "Real observed target label disagrees between this notebook's fresh load and Problem 4's saved "
        "severity_scores_holdout.csv for at least one customer -- the two notebooks are not looking at the "
        "same holdout split.\nFix: re-run Problem 1's Notebook 04 once, then Problem 4's Notebook 27, then this notebook."
    )
ecl_df = ecl_df.drop(columns=["p4_target"])
TIER_ORDER = ECL_POLICY["lgd_by_tier"]["tier_order"]
ecl_df["severity_tier"] = pd.Categorical(ecl_df["severity_tier"], categories=TIER_ORDER, ordered=True)

print(f"Holdout customers scored here (real)         : {len(pd_df):,}")
print(f"Matched 1-to-1 with Problem 4's real tiers    : {len(ecl_df):,}")
print(f"Target label cross-check (this run vs Problem 4): agrees for all matched customers")
print(ecl_df["severity_tier"].value_counts().reindex(TIER_ORDER).to_string())
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: IFRS 9 STAGING -- OUTCOME-FREE RUBRIC (REAL PD + REAL TIER ONLY)
# =============================================================================
_section("SECTION 6: IFRS 9 Staging -- Outcome-Free Rubric (Real PD + Real Tier Only)")

# --- Every input here is real and known BEFORE default: PD (Section 4) and
#     severity tier (Problem 4). The real `target` column is NOT referenced
#     anywhere in this section -- it is used only in Section 10, afterward,
#     purely to VALIDATE the result. Stage 3 is checked first so it takes
#     precedence over Stage 2 for a customer who would otherwise qualify for both. ---
_sc = ECL_POLICY["staging_criteria"]
_portfolio_avg_pd = float(ecl_df["pd_12m"].mean())
_sicr_threshold = _portfolio_avg_pd * _sc["sicr_pd_multiple"]
_stage3_pd_threshold = _sc["stage3_pd_threshold"]

_pd = ecl_df["pd_12m"].to_numpy()
_tier = ecl_df["severity_tier"].astype(str).to_numpy()
_is_severe = _tier == "Severe"
_is_elevated_tier = np.isin(_tier, ["Moderate Severity", "Severe"])

ifrs9_stage = np.where(
    _is_severe & (_pd > _stage3_pd_threshold), 3,
    np.where((_pd > _sicr_threshold) | _is_elevated_tier, 2, 1)
)
ecl_df["ifrs9_stage"] = ifrs9_stage

stage_counts = ecl_df["ifrs9_stage"].value_counts().sort_index()
print(f"Portfolio average PD (this run, real)  : {_portfolio_avg_pd:.4f}")
print(f"Stage 2 SICR threshold ({_sc['sicr_pd_multiple']}x average): {_sicr_threshold:.4f}")
print(f"Stage 3 absolute PD threshold (ASSUMPTION): {_stage3_pd_threshold}")
print(f"\nStaging outcome (holdout population, {len(ecl_df):,} customers, outcome-free):")
for stage in (1, 2, 3):
    _n = int(stage_counts.get(stage, 0))
    _label = {1: "Performing, no SICR", 2: "Significant Increase in Credit Risk (SICR)",
              3: "Credit-Impaired (real PD + real tier, NOT the known outcome)"}[stage]
    print(f"  Stage {stage} ({_label}): {_n:,} customers ({_n / len(ecl_df):.2%})")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: LIFETIME PD & FORWARD-LOOKING MACRO OVERLAY
# =============================================================================
_section("SECTION 7: Lifetime PD & Forward-Looking Macro Overlay")

# --- Lifetime PD: ASSUMPTION multiplier on 12-month PD (Notebook 30's policy;
#     no PD term-structure data exists to fit a real survival curve).
#     Macro overlay: probability-weighted blend across the 3 stated scenarios,
#     applied to BOTH the 12-month and lifetime PD, per Notebook 30's policy. ---
_lifetime_mult = ECL_POLICY["lifetime_pd"]["lifetime_pd_multiplier"]
pd_lifetime = np.clip(ecl_df["pd_12m"].to_numpy() * _lifetime_mult, 0.0, 1.0)
ecl_df["pd_lifetime"] = pd_lifetime

_scenarios = ECL_POLICY["macro_overlay"]["scenarios"]
pd_12m_macro = np.zeros(len(ecl_df))
pd_lifetime_macro = np.zeros(len(ecl_df))
for _s in _scenarios:
    pd_12m_macro += _s["probability"] * np.clip(ecl_df["pd_12m"].to_numpy() * _s["pd_multiplier"], 0.0, 1.0)
    pd_lifetime_macro += _s["probability"] * np.clip(pd_lifetime * _s["pd_multiplier"], 0.0, 1.0)
ecl_df["pd_12m_macro_adj"] = pd_12m_macro
ecl_df["pd_lifetime_macro_adj"] = pd_lifetime_macro

print(f"Lifetime PD multiplier (ASSUMPTION)      : {_lifetime_mult}x")
print(f"Macro scenarios (ASSUMPTION, probability-weighted): "
      + ", ".join(f"{s['scenario']}={s['probability']:.0%}@{s['pd_multiplier']}x" for s in _scenarios))
print(f"Portfolio avg PD_12m before/after macro overlay : {ecl_df['pd_12m'].mean():.4f} -> "
      f"{pd_12m_macro.mean():.4f}")
print(f"Portfolio avg PD_lifetime before/after macro overlay: {pd_lifetime.mean():.4f} -> "
      f"{pd_lifetime_macro.mean():.4f}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: DISCOUNT TO PRESENT VALUE (IFRS9 §5.5.17)
# =============================================================================
_section("SECTION 8: Discount to Present Value (IFRS9 \u00a75.5.17)")

_dr = ECL_POLICY["discount_rate"]
_annual_rate = _dr["annual_rate"]
DF_STAGE1 = 1.0 / ((1.0 + _annual_rate) ** _dr["stage1_discount_period_years"])
DF_STAGE23 = 1.0 / ((1.0 + _annual_rate) ** _dr["stage23_avg_remaining_life_years"])

print(f"Discount rate (ASSUMPTION, annual EIR proxy): {_annual_rate:.1%}")
print(f"Stage 1 discount factor (t={_dr['stage1_discount_period_years']}yr)  : {DF_STAGE1:.4f}")
print(f"Stage 2/3 & CECL discount factor (t={_dr['stage23_avg_remaining_life_years']}yr): {DF_STAGE23:.4f}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: EXPECTED CREDIT LOSS -- IFRS 9 (STAGED) AND CECL (LIFETIME, NO STAGING)
# =============================================================================
_section("SECTION 9: Expected Credit Loss -- IFRS 9 (Staged) and CECL (Lifetime, No Staging)")

EAD = ECL_POLICY["ead_per_account_usd"]
LGD_BY_TIER = ECL_POLICY["lgd_by_tier"]["values"]
lgd_per_customer = ecl_df["severity_tier"].astype(str).map(LGD_BY_TIER).to_numpy()
ecl_df["lgd_tier"] = lgd_per_customer

# --- IFRS9: Stage 1 uses 12-month macro-adjusted PD, discounted at the Stage 1
#     convention. Stage 2/3 use LIFETIME macro-adjusted PD, discounted at the
#     lifetime convention. Unlike Notebook 08, Stage 3 here does NOT force
#     PD=1.0 from the known outcome -- it uses the real computed lifetime PD,
#     which is already high by construction (Stage 3 requires PD_12m > 0.50). ---
ecl_ifrs9 = np.where(
    ecl_df["ifrs9_stage"].to_numpy() == 1,
    ecl_df["pd_12m_macro_adj"].to_numpy() * lgd_per_customer * EAD * DF_STAGE1,
    ecl_df["pd_lifetime_macro_adj"].to_numpy() * lgd_per_customer * EAD * DF_STAGE23,
)
ecl_df["ecl_ifrs9_usd"] = ecl_ifrs9

# --- CECL: lifetime ECL for the ENTIRE portfolio from day one, no staging --
#     the defining difference from IFRS9's 3-stage approach. Same lifetime PD,
#     tier-LGD, EAD, and discount convention as IFRS9 Stage 2/3, applied to
#     every customer regardless of stage. ---
ecl_cecl = ecl_df["pd_lifetime_macro_adj"].to_numpy() * lgd_per_customer * EAD * DF_STAGE23
ecl_df["ecl_cecl_usd"] = ecl_cecl

TOTAL_ECL_IFRS9_USD = float(ecl_df["ecl_ifrs9_usd"].sum())
TOTAL_ECL_CECL_USD = float(ecl_df["ecl_cecl_usd"].sum())
print(f"Total IFRS9 ECL (staged, holdout, this run): ${TOTAL_ECL_IFRS9_USD:,.0f}")
print(f"Total CECL ECL (lifetime-for-all, holdout) : ${TOTAL_ECL_CECL_USD:,.0f}")
print(f"CECL vs IFRS9 gap (lifetime-for-all effect): ${TOTAL_ECL_CECL_USD - TOTAL_ECL_IFRS9_USD:,.0f} "
      f"({(TOTAL_ECL_CECL_USD / TOTAL_ECL_IFRS9_USD - 1):+.1%})")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: POST-HOC VALIDATION -- STAGE RANK-ORDERING AGAINST REAL OUTCOMES
# =============================================================================
_section("SECTION 10: Post-Hoc Validation -- Stage Rank-Ordering Against Real Outcomes")

# --- The real `target` label is used HERE ONLY, to check that outcome-free
#     staging still rank-orders with what actually happened -- never to assign
#     a stage (Section 6 never touched it). This is a validation-after-the-fact,
#     the same honest pattern Notebook 27 used for its severity tiers. ---
stage_validation = ecl_df.groupby("ifrs9_stage").agg(
    n_customers=("customer_ID", "size"), observed_default_rate=("target", "mean"),
    avg_pd_12m=("pd_12m", "mean"), total_ecl_ifrs9_usd=("ecl_ifrs9_usd", "sum"),
).reset_index()
stage_validation["pct_of_portfolio"] = stage_validation["n_customers"] / len(ecl_df)
print(stage_validation.to_string(index=False))

_rates = stage_validation.sort_values("ifrs9_stage")["observed_default_rate"].to_numpy()
_is_monotonic = bool(np.all(np.diff(_rates) > 0))
print(f"\nStrict monotonicity (Stage1 < Stage2 < Stage3 real default rate): {_is_monotonic}")
if ECL_POLICY["kpi_targets"]["require_strict_monotonicity_default_rate_by_stage"] and not _is_monotonic:
    raise RuntimeError(
        "IFRS9 stage real observed default rate is NOT strictly monotonic -- the outcome-free staging "
        "rubric does not rank-order with real outcomes on this holdout split. Fix: revisit Notebook 30's "
        "staging thresholds (sicr_pd_multiple / stage3_pd_threshold) and re-run Notebooks 30-31."
    )

_stage3_pct = float(stage_validation.loc[stage_validation["ifrs9_stage"] == 3, "pct_of_portfolio"].iloc[0]) * 100 \
    if 3 in stage_validation["ifrs9_stage"].values else 0.0
_kpi = ECL_POLICY["kpi_targets"]
_stage3_in_range = _kpi["min_stage3_population_pct"] <= _stage3_pct <= _kpi["max_stage3_population_pct"]
print(f"Stage 3 population share: {_stage3_pct:.2f}% "
      f"(sanity range {_kpi['min_stage3_population_pct']}-{_kpi['max_stage3_population_pct']}%, "
      f"{'within range' if _stage3_in_range else 'OUTSIDE range -- reported honestly either way'})")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: STANDARD COMPARISON -- NOTEBOOK 08 FLAT-LGD vs IFRS9 TIER-LGD vs CECL
# =============================================================================
_section("SECTION 11: Standard Comparison -- Notebook 08 Flat-LGD vs IFRS9 Tier-LGD vs CECL")

comparison_rows = [
    {"methodology": "Notebook 08: Basel/IFRS9, flat 45% LGD, outcome-based Stage 3",
     "total_ecl_usd": NB08_SUMMARY["total_ecl_usd"]},
    {"methodology": "Notebook 31: IFRS9 staged, tier-LGD, outcome-free staging",
     "total_ecl_usd": TOTAL_ECL_IFRS9_USD},
    {"methodology": "Notebook 31: CECL lifetime-for-all, tier-LGD, no staging",
     "total_ecl_usd": TOTAL_ECL_CECL_USD},
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_df["delta_vs_notebook_08_usd"] = comparison_df["total_ecl_usd"] - NB08_SUMMARY["total_ecl_usd"]
print(comparison_df.to_string(index=False))

_delta_ifrs9 = TOTAL_ECL_IFRS9_USD - NB08_SUMMARY["total_ecl_usd"]
_interpretation = (
    f"Moving from Notebook 08's flat-LGD Basel exercise to this notebook's tier-differentiated, "
    f"outcome-free-staged IFRS9 ECL changes recognized loss by ${_delta_ifrs9:,.0f} "
    f"({'an increase' if _delta_ifrs9 > 0 else 'a decrease'} in provisioning) on the same real holdout "
    f"population. The CECL lifetime-for-all figure is "
    f"{'higher' if TOTAL_ECL_CECL_USD > TOTAL_ECL_IFRS9_USD else 'lower'} than the IFRS9 staged figure by "
    f"${abs(TOTAL_ECL_CECL_USD - TOTAL_ECL_IFRS9_USD):,.0f} -- the real, expected effect of CECL requiring "
    "lifetime losses for the entire portfolio from day one, versus IFRS9 reserving lifetime losses only "
    "for Stage 2/3 accounts."
)
print(f"\n{_interpretation}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: INLINE CHARTS
# =============================================================================
_section("SECTION 12: Inline Charts")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "gold": "#B08D57", "surface": "#FFFFFF"}

fig1, ax1 = plt.subplots(figsize=(7.5, 5), dpi=150)
_stage_labels = [f"Stage {s}" for s in stage_validation["ifrs9_stage"]]
ax1.bar(_stage_labels, stage_validation["total_ecl_ifrs9_usd"],
        color=[VIZ["muted"], VIZ["gold"], VIZ["accent"]][:len(stage_validation)])
ax1.set_ylabel("IFRS9 ECL, USD (real, this run)")
ax1.set_title("Problem 3: IFRS9 Expected Credit Loss by Stage (Outcome-Free Staging)")
fig1.tight_layout()
chart1_path = PILLAR_DIRS["p3_modeling"] / "ecl_by_stage_chart.png"
fig1.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(7.5, 5), dpi=150)
ax2.bar(comparison_df["methodology"].str.replace("Notebook ", "NB", regex=False),
        comparison_df["total_ecl_usd"], color=[VIZ["muted"], VIZ["gold"], VIZ["accent"]])
ax2.set_ylabel("Total ECL, USD (real, this run)")
ax2.set_title("Problem 3: Flat-LGD vs Tier-LGD IFRS9 vs CECL -- Total ECL")
ax2.tick_params(axis="x", labelsize=7)
fig2.tight_layout()
chart2_path = PILLAR_DIRS["p3_modeling"] / "ecl_standard_comparison_chart.png"
fig2.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig2)

print(f"\u2705 Saved -> {chart1_path.name}, {chart2_path.name} (this problem's 02_ECL_Modeling folder)")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: SAVE ARTIFACTS
# =============================================================================
_section("SECTION 13: Save Artifacts")

ecl_by_customer_path = PILLAR_DIRS["p3_modeling"] / "ecl_by_customer.csv"
ecl_df[["customer_ID", "pd_12m", "pd_lifetime", "severity_tier", "lgd_tier", "ifrs9_stage",
        "ecl_ifrs9_usd", "ecl_cecl_usd", "target"]].to_csv(ecl_by_customer_path, index=False)

stage_summary_path = PILLAR_DIRS["p3_modeling"] / "ecl_stage_summary.csv"
stage_validation.to_csv(stage_summary_path, index=False)

comparison_path = PILLAR_DIRS["p3_modeling"] / "ecl_standard_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print(f"\u2705 Saved -> {ecl_by_customer_path.name}, {stage_summary_path.name}, {comparison_path.name}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: VERIFICATION
# =============================================================================
_section("SECTION 14: Verification")

_checks_passed = True


def _check(label, condition, detail="", hard=True):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        if hard:
            _checks_passed = False
        _mark = "\u274c" if hard else "\u26a0\ufe0f"
        print(f"{_mark} {label}  {detail}")


_check("Stage real default rate strictly monotonic (hard requirement)", _is_monotonic, hard=True)
_check("Every customer joined to a real Problem 4 severity tier", ecl_df["severity_tier"].notna().all(), hard=True)
_check("No NaN in IFRS9 or CECL ECL", ecl_df[["ecl_ifrs9_usd", "ecl_cecl_usd"]].notna().all().all(), hard=True)
_check("CECL total ECL >= IFRS9 total ECL (lifetime-for-all cannot recognize less)",
       TOTAL_ECL_CECL_USD >= TOTAL_ECL_IFRS9_USD, hard=True)
_check("Stage 3 population within stated sanity range", _stage3_in_range, hard=False,
       detail=f"(measured {_stage3_pct:.2f}%)")

_expected_files = [ecl_by_customer_path, stage_summary_path, comparison_path, chart1_path, chart2_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 31 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 31 hard checks passed.")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: WRITE NOTEBOOK 31 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 15: Write Notebook 31 Summary Artifact")

notebook_31_summary = {
    "notebook": "31_ecl_modeling", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 3, "problem_name": "Expected Credit Loss (IFRS9/CECL)",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning",
    "n_holdout_customers": int(len(ecl_df)), "champion_pd_model": CHAMPION_NAME,
    "monotonic_validated": bool(_is_monotonic),
    "total_ecl_ifrs9_usd": TOTAL_ECL_IFRS9_USD, "total_ecl_cecl_usd": TOTAL_ECL_CECL_USD,
    "total_ecl_notebook_08_flat_usd": NB08_SUMMARY["total_ecl_usd"],
    "stage_counts": {int(k): int(v) for k, v in stage_counts.items()},
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb31_summary_path = ARTIFACTS_DIR / "notebook_31_summary.json"
with open(nb31_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_31_summary, f, indent=2)
print(f"\u2705 Saved -> {nb31_summary_path.name} (this problem's artifacts folder)")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 16: Notebook 31 Complete -- Handoff to Notebook 32")

print("NOTEBOOK 31: ECL MODELING -- COMPLETE")
print(f"  Holdout customers scored (real)       : {len(ecl_df):,}")
print(f"  Stage rank-ordering validated (real)  : {_is_monotonic}")
print(f"  Total IFRS9 ECL (staged, real)         : ${TOTAL_ECL_IFRS9_USD:,.0f}")
print(f"  Total CECL ECL (lifetime-for-all, real): ${TOTAL_ECL_CECL_USD:,.0f}")
print(f"  Notebook 08 flat-LGD baseline (real)   : ${NB08_SUMMARY['total_ecl_usd']:,.0f}")
print(f"  Files produced                        : {len(_expected_files) + 1}")
for _p in _expected_files + [nb31_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                          : 32_validation_deployment.ipynb")
print("\n\u2705 Ready to proceed.")
